# linalg-solve-batched composite — cx2: stack columns into batched matrix, then linalg.solve in one shot

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `linalg-solve-batched`, `stack-vs-cat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "linalg-solve-batched"
DD_ATOM_IDS = ["linalg-solve-batched", "stack-vs-cat"]
DD_SUBTOPICS = ["PyTorch: Batched linalg.solve", "PyTorch: stack vs cat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The batched-solve atom expects shape `(K, n, n)` for `A` and `(K, n)` for `b`. But upstream you often hold the column vectors of each `A_k` SEPARATELY — three `(K, n)` tensors that need to be fused into one `(K, n, n)` tensor before the solve.

`stack-vs-cat` answers the dispatch question: same rank as inputs would need `cat`, but here we need an EXTRA axis (one slot per column), so it's `t.stack(cols, dim=-1)`. The trailing-axis insertion lines up exactly with `linalg.solve`'s `(K, n, n)` shape contract — last axis indexes the columns, second-to-last indexes the row entries, first axis is the batch.

This composition is the meat of the ARENA Day 1 ray-triangle pipeline: build columns from geometry, stack to a batched matrix, batched solve, done — no Python loops.

### Composite Exercise — stack columns into batched matrix, then linalg.solve in one shot

**Atoms exercised together**: `linalg-solve-batched`, `stack-vs-cat`

Implement `cx2_solve_from_columns(cols, b)` that solves `K` independent `n x n` linear systems given the columns of each `A_k` separately.

- `cols` is a list of `n` tensors, each of shape `(K, n)` — `cols[j]` is column `j` of each system's matrix.
- `b` has shape `(K, n)` — the right-hand sides.

1. **Stack** the columns into `A` of shape `(K, n, n)` with `t.stack(cols, dim=-1)`. (Last-axis insertion: each column becomes a slot along the new trailing axis.) Do NOT use `cat` — same-rank concatenation would produce `(K, n*n)`, not `(K, n, n)`.
2. **Batched solve** with `t.linalg.solve(A, b)` — one call, no loop.

Return the `(K, n)` solution tensor.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx2_solve_from_columns(cols, b):
    raise NotImplementedError

def _test_cx2():
    # Case A: K=2 hand-picked 2x2 systems.
    # System 0: A = [[1,0],[0,1]], b = [3,-1] -> x = [3,-1]
    # System 1: A = [[2,1],[1,3]], b = [5, 5] -> x = [2, 1]
    col0 = t.tensor([[1.0, 0.0], [2.0, 1.0]])  # K=2, n=2: col 0 of each system
    col1 = t.tensor([[0.0, 1.0], [1.0, 3.0]])  # col 1 of each system
    b = t.tensor([[3.0, -1.0], [5.0, 5.0]])
    x = cx2_solve_from_columns([col0, col1], b)
    assert tuple(x.shape) == (2, 2), f'expected (2,2), got {tuple(x.shape)}'
    assert t.allclose(x[0], t.tensor([3.0, -1.0]), atol=1e-5)
    assert t.allclose(x[1], t.tensor([2.0, 1.0]), atol=1e-5)

    # Case B: K=8 random 3x3 systems — cross-check by recomputing A @ x == b.
    rng = t.Generator().manual_seed(1)
    K, n = 8, 3
    # Build well-conditioned A as I + small perturbation.
    I3 = t.eye(3).unsqueeze(0).expand(K, n, n).clone()
    A_full = I3 + 0.1 * t.randn(K, n, n, generator=rng)
    b_r = t.randn(K, n, generator=rng)
    cols_r = [A_full[:, :, j] for j in range(n)]
    x_r = cx2_solve_from_columns(cols_r, b_r)
    assert tuple(x_r.shape) == (K, n)
    recon = t.einsum('kij,kj->ki', A_full, x_r)
    assert t.allclose(recon, b_r, atol=1e-4), f'A@x != b, max diff {(recon-b_r).abs().max()}'

    # Case C: ray-triangle-style 3-column build, then solve.
    # Verify the composition matches a manual stack + solve.
    K2, n2 = 4, 3
    rng2 = t.Generator().manual_seed(2)
    c0 = t.randn(K2, n2, generator=rng2)
    c1 = t.randn(K2, n2, generator=rng2)
    c2 = t.randn(K2, n2, generator=rng2)
    # Add a diagonal boost to keep matrices well-conditioned.
    boost = t.stack([c0, c1, c2], dim=-1) + 2.0 * t.eye(n2)
    c0b = boost[:, :, 0]; c1b = boost[:, :, 1]; c2b = boost[:, :, 2]
    b2 = t.randn(K2, n2, generator=rng2)
    x2 = cx2_solve_from_columns([c0b, c1b, c2b], b2)
    ref = t.linalg.solve(t.stack([c0b, c1b, c2b], dim=-1), b2)
    assert t.allclose(x2, ref, atol=1e-5)
    _dd_passed.add('cx2')

_test_cx2()

<details><summary>Show solution — cx2</summary>

```python
def cx2_solve_from_columns(cols, b):
    # stack-vs-cat: NEW trailing axis (one slot per column) -> stack(dim=-1).
    A = t.stack(cols, dim=-1)        # (K, n, n)
    # linalg-solve-batched: single C-level batched call, no Python loop.
    return t.linalg.solve(A, b)      # (K, n)
```

Two atoms, two lines. The `stack(dim=-1)` choice is what makes this a `stack-vs-cat` exercise — `cat(dim=-1)` would silently produce shape `(K, n*n)` which would then either crash `linalg.solve` (rank mismatch) or, worse, succeed-with-broadcasting in a way that gives garbage answers.

Once the shapes are right, `t.linalg.solve(A, b)` fuses K LU factorizations into one BLAS call. The leading batch dim is just a convention — solve treats it as 'do this n x n problem K times', no loop required.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx2',
        'subtopics': ["PyTorch: Batched linalg.solve", "PyTorch: stack vs cat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()